# Modul 4: Estimasi Parameter ARMA & Kestabilan Bidang-Z
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 4 (Minggu 4):** Menyelesaikan persamaan Yule-Walker untuk AR($p$), menguji kestabilan akar karakteristik pada lingkaran satuan bidang-$Z$, dan memilih ordo ARMA($p,q$) berbasis AIC/BIC.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')
plt.rcParams['axes.grid'] = True

## 1. Simulasi Sinyal Getaran Generator (Proses AR(2) Stabil)

In [ ]:
rng = np.random.default_rng(101)
n = 400
phi1_true, phi2_true = 0.75, -0.35
y = np.zeros(n)
eps = rng.normal(0, 1.0, n)
for t in range(2, n):
    y[t] = phi1_true * y[t - 1] + phi2_true * y[t - 2] + eps[t]

signal = pd.Series(y, name='vibration')
signal.plot(figsize=(12, 3.5), title='Sinyal Getaran Generator (Simulasi AR(2))')
plt.show()

## 2. Estimasi Analitik Yule-Walker & Plot Akar Kestabilan Bidang-Z

In [ ]:
# Persamaan Yule-Walker: R * phi = r
r_vals = acf(signal, nlags=2)
R = np.array([[1.0, r_vals[1]], [r_vals[1], 1.0]])
r_vec = np.array([r_vals[1], r_vals[2]])
phi_yw = np.linalg.solve(R, r_vec)
print(f"Parameter Asli       : phi_1 = {phi1_true:.4f}, phi_2 = {phi2_true:.4f}")
print(f"Estimasi Yule-Walker : phi_1 = {phi_yw[0]:.4f}, phi_2 = {phi_yw[1]:.4f}")

# Fit MLE dengan ARIMA(2,0,0) dan uji akar polinomial karakteristik
fit_ar2 = ARIMA(signal, order=(2, 0, 0)).fit()
ar_roots = fit_ar2.arroots          # Akar dari 1 - phi_1*z - phi_2*z^2 = 0 (harus |z| > 1)
inv_roots = 1.0 / ar_roots          # Pole pada bidang-Z konvensional (harus |pole| < 1)
print("Akar polinomial lag (harus |root| > 1):", np.abs(ar_roots).round(4))
print("Stabil / Kausal:", np.all(np.abs(ar_roots) > 1.0))

fig, ax = plt.subplots(figsize=(5.5, 5.5))
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), 'r--', label='Lingkaran Satuan |z|=1')
ax.scatter(inv_roots.real, inv_roots.imag, s=90, color='navy', label='Inverse AR Roots (Poles)')
ax.axhline(0, color='gray', lw=0.7)
ax.axvline(0, color='gray', lw=0.7)
ax.set_aspect('equal')
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_title('Diagram Kestabilan Pole Bidang-Z (Stabil jika di dalam lingkaran)')
ax.legend()
plt.show()

## 3. Grid-Search Seleksi Ordo ARMA(p,q) Berbasis AIC & BIC

In [ ]:
rows = []
for p in range(4):
    for q in range(3):
        if p == 0 and q == 0:
            continue
        res = ARIMA(signal, order=(p, 0, q)).fit()
        rows.append({'order': f'ARMA({p},{q})', 'AIC': res.aic, 'BIC': res.bic})

df_ic = pd.DataFrame(rows).sort_values('AIC')
print(df_ic.to_string(index=False))

## 4. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Bandingkan model terbaik menurut **AIC** vs. **BIC**. Mengapa BIC cenderung memilih model lebih parsimonius?
2. **[C5]** Ubah parameter simulasi menjadi $\phi_1 = 0.8, \phi_2 = 0.25$ lalu periksa lokasi *pole* terhadap lingkaran satuan. Apa yang terjadi pada sinyal?